# Mini-Grid URL Collector

Single-purpose notebook: given a **country name**, run a set of web searches
about mini-grid projects in that country, collect every unique URL found,
and write them to a plain text file (one URL per line).

This notebook does **not** fetch pages or extract structured data — that is
handled by `pipeline.py`. It only discovers and persists candidate source URLs.

Relevance is judged by Gemini (with a keyword fallback if no API key is set)
to filter out results that are about the wrong country or an unrelated topic.

In [20]:
import json
import os
import re
import time

from ddgs import DDGS
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

True

In [21]:
# ── Config ────────────────────────────────────────────────────────────────

COUNTRY = "San Marino"      # <- set the target country here  (iraq, ), seychelles, belgium, denmark, finland, france, iceland, luxembourg, monaco, norway, sweden, switzerland
NUM_SEARCHES = 20         # how many queries to run (capped to len(queries))
MAX_RESULTS_PER_QUERY = 10
SLEEP_BETWEEN_QUERIES = 1.5  # seconds, avoid rate limiting

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")  # reads from .env — see GEMINI_API_KEY=... there
GEMINI_MODEL = "gemini-2.5-flash"
USE_LLM_FILTER = bool(GEMINI_API_KEY) and GEMINI_API_KEY != "your_api_key_here"

FRANCOPHONE = {
    "chad", "niger", "mali", "senegal", "burkina faso", "guinea",
    "cameroon", "madagascar", "benin", "togo", "gabon", "côte d'ivoire",
    "cote d'ivoire", "drc", "congo", "mauritania", "comoros",
}


def build_queries(country: str) -> list[str]:
    queries = [
       f"{country} progetto mini-grid",
f"{country} progetto microgrid elettrificazione rurale",
f"{country} mini grid solare elettrificazione rurale",
f"{country} off-grid mini-grid sviluppatore rapporto",
f"{country} mini-grid messa in servizio connessioni abitazioni",
f"site:irena.org {country} mini-grid",
f"site:esmap.org {country} mini-grid",
f"site:africa-energy-portal.org {country} mini-grid",
f"{country} mini-grid sviluppatore privato BBOXX Engie PowerGen Ignite Husk",
f"{country} mini-rete solare elettrificazione rurale",
f"{country} accesso energia off-grid rapporto ministero",
f"{country} mini-rete elettrica",
f"{country} progetti mini reti",


        f"{country} mini-grid project",
        f"{country} microgrid project rural electrification",
        f"{country} mini grid solar electrification rural",
        f"{country} off-grid mini-grid developer report",
        f"{country} mini-grid commissioning connections households",
        f"site:irena.org {country} mini-grid",
        f"site:esmap.org {country} mini-grid",
        f"site:africa-energy-portal.org {country} mini-grid",
        f"{country} mini-grid private developer BBOXX Engie PowerGen Ignite Husk",  
        f"{country} mini-réseau solaire électrification rurale",
        f"{country} accès énergie hors-réseau rapport ministère",
        f"{country} mini-réseau électrique",
        f"{country} projets mini réseaux",
#         f"{country} مشروع شبكة كهربائية صغيرة",
# f"{country} شبكة كهربائية صغيرة كهربة ريفية",
# f"{country} شبكة كهربائية صغيرة كهرباء شمسية ريفية",
# f"{country} تقرير مطور شبكات كهربائية صغيرة خارج الشبكة",
# f"{country} تشغيل شبكة كهربائية صغيرة توصيلات منازل",
# f"site:irena.org {country} شبكة كهربائية صغيرة",
# f"site:esmap.org {country} شبكة كهربائية صغيرة",
# f"site:africa-energy-portal.org {country} شبكة كهربائية صغيرة",
# f"{country} شبكة كهربائية صغيرة مطور خاص BBOXX Engie PowerGen Ignite Husk",
# f"{country} شبكة كهربائية صغيرة شمسية كهربة ريفية",
# f"{country} الوصول إلى الطاقة خارج الشبكة تقرير وزارة",
# f"{country} شبكة كهربائية صغيرة",
# f"{country} مشاريع الشبكات الكهربائية الصغيرة",

        # f"{country} mini-rețea OR mini-rețele energie solară proiect sat capacitate kW",
        # f"{country} mini-rețea OR mini-rețele solar fotovoltaic proiect comunitate capacitate kW",
        # f"{country} sistem izolat OR rețea izolată generare solară comunitate rurală kW",
        # f"{country} generare distribuită regenerabilă comunitate izolată proiect pilot",
        # f"{country} centrală solară comunitară hibridă diesel baterie kWp sat",
        # f"{country} electrificare rurală off-grid mini-rețea proiect ministerul energiei",
        # f"{country} program național electrificare rurală comunități izolate off-grid",
        # f"{country} concesiune OR licitație mini-rețea zone neinterconectate",
        # f"{country} mini-rețea proiect Banca Mondială BID GIZ PNUD finanțare grant",
        # f"{country} electrificare rurală regenerabilă cooperare internațională comunități indigene",
        # f"{country} projetos mini-grid, off-grid, aldeias remotas e aldeias inteligentes Portugal",
        # f"{country} pienverkko OR pienverkot aurinkosähkö projekti kylä kapasiteetti kW",
# f"{country} pienverkko OR pienverkot aurinko fotovoltainen projekti yhteisö kapasiteetti kW",
# f"{country} erillinen järjestelmä OR erillinen verkko aurinkoenergia tuotanto maaseutuyhteisö kW",
# f"{country} hajautettu uusiutuva energiantuotanto eristetty yhteisö pilottihanke",
# f"{country} yhteisöllinen hybridi aurinkovoimala diesel akku kWp kylä",
# f"{country} maaseudun sähköistys off-grid pienverkko hanke energiaministeriö",
# f"{country} kansallinen ohjelma maaseudun sähköistys eristetyt yhteisöt off-grid",
# f"{country} toimilupa OR tarjouskilpailu pienverkko liittämättömät alueet",
# f"{country} pienverkko hanke Maailmanpankki BID GIZ UNDP rahoitus avustus",
# f"{country} maaseudun uusiutuva sähköistys kansainvälinen yhteistyö alkuperäisyhteisöt",
# f"{country} pienverkko hankkeet off-grid syrjäiset kylät älykkäät kylät",


      

    ]
    if country.lower() in FRANCOPHONE:
        queries += [
            f"{country} mini-réseau solaire électrification rurale",
            f"{country} accès énergie hors-réseau rapport ministère",
        ]
    return queries




    #   f"{country} mini-rede OR mini-redes energia solar projeto aldeia capacidade kW",
    #     f"{country} minirred OR mini-red OR microrred solar fotovoltaica proyecto comunidad capacidad kW",
    #     f"{country} sistema aislado OR red aislada generación solar comunidad rural kW",
    #     f"{country} generación distribuida renovable comunidad aislada proyecto piloto",
    #     f"{country} planta solar comunitaria híbrida diésel batería kWp aldea",
    #     f"{country} electrificación rural fuera de la red minirred proyecto ministerio energía",
    #     f"{country} programa nacional electrificación rural comunidades aisladas off-grid",
    #     f"{country} concesión OR licitación minirred zonas no interconectadas",
    #     f"{country} minirred proyecto Banco Mundial BID GIZ PNUD financiamiento subvención",
    #     f"{country} electrificación rural renovable cooperación internacional comunidades indígenas",

In [22]:
# ── Relevance filtering ──────────────────────────────────────────────────────

_llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0, google_api_key=GEMINI_API_KEY) \
    if USE_LLM_FILTER else None


def keyword_relevant(country: str, text: str) -> bool:
    """Fallback heuristic: keep a result only if the country name appears in it."""
    return country.lower() in text.lower()


def llm_judge_relevance(country: str, results: list[dict]) -> list[bool]:
    """
    Ask Gemini to judge, for a batch of search results, whether each one is
    genuinely about mini-grid/microgrid energy projects in `country` specifically
    (not another country, and not an unrelated topic that happens to mention it).
    Falls back to the keyword heuristic if the LLM call fails or is disabled.
    """
    if not results:
        return []
    if _llm is None:
        return [
            keyword_relevant(country, f"{r.get('title','')} {r.get('body','')} {r.get('href','')}")
            for r in results
        ]

    items = "\n".join(
        f"{i+1}. Title: {r.get('title','')} | Snippet: {r.get('body','')[:300]} | URL: {r.get('href','')}"
        for i, r in enumerate(results)
    )
    prompt = (
        f"You are filtering web search results for relevance to mini-grid or microgrid "
        f"energy projects specifically in {country} (not any other country).\n\n"
        f"For each numbered result below, decide if it is genuinely about {country} AND "
        f"about mini-grids, microgrids, off-grid electrification, or rural energy access.\n"
        f"A result about the right topic but the wrong country must be marked false.\n\n"
        f"{items}\n\n"
        f"Return ONLY a JSON array of exactly {len(results)} booleans, one per result, "
        f"in order. No explanation, no markdown."
    )
    try:
        response = _llm.invoke(prompt)
        raw = response.content.strip()
        raw = re.sub(r"^```[a-z]*\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
        verdicts = json.loads(raw)
        if isinstance(verdicts, list) and len(verdicts) == len(results):
            return [bool(v) for v in verdicts]
        print("  [!] LLM returned malformed verdict list, falling back to keyword filter")
    except Exception as e:
        print(f"  [!] LLM relevance check failed ({e}), falling back to keyword filter")

    return [
        keyword_relevant(country, f"{r.get('title','')} {r.get('body','')} {r.get('href','')}")
        for r in results
    ]

In [23]:
# ── URL collection ────────────────────────────────────────────────────────

def collect_urls(country: str, num_searches: int = NUM_SEARCHES) -> list[str]:
    queries = build_queries(country)[:num_searches]
    seen = set()
    ordered_urls = []
    dropped = 0

    for i, query in enumerate(queries, 1):
        print(f"[{i}/{len(queries)}] Searching: {query}")
        try:
            results = list(DDGS().text(query, max_results=MAX_RESULTS_PER_QUERY))
            fresh = [r for r in results if r.get("href") and r["href"] not in seen]
            verdicts = llm_judge_relevance(country, fresh)

            new_count = 0
            for r, keep in zip(fresh, verdicts):
                url = r["href"]
                if keep:
                    seen.add(url)
                    ordered_urls.append(url)
                    new_count += 1
                else:
                    seen.add(url)  # don't re-evaluate the same URL in a later query
                    dropped += 1
            print(f"  → {len(results)} result(s), {new_count} new relevant URL(s)")
        except Exception as e:
            print(f"  [!] Search error: {e}")
        time.sleep(SLEEP_BETWEEN_QUERIES)

    print(f"\nFiltered out {dropped} irrelevant result(s)")
    return ordered_urls

In [24]:
# ── Run & save ────────────────────────────────────────────────────────────

urls = collect_urls(COUNTRY)

country_slug = re.sub(r"[^a-z0-9]+", "_", COUNTRY.lower()).strip("_")
output_path = f"urls_{country_slug}.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write("\n".join(urls))

print(f"\nSaved {len(urls)} unique URL(s) to {output_path}")

[1/20] Searching: San Marino progetto mini-grid
  → 10 result(s), 4 new relevant URL(s)
[2/20] Searching: San Marino progetto microgrid elettrificazione rurale
  → 10 result(s), 1 new relevant URL(s)
[3/20] Searching: San Marino mini grid solare elettrificazione rurale
  → 10 result(s), 0 new relevant URL(s)
[4/20] Searching: San Marino off-grid mini-grid sviluppatore rapporto
  → 10 result(s), 0 new relevant URL(s)
[5/20] Searching: San Marino mini-grid messa in servizio connessioni abitazioni
  → 10 result(s), 0 new relevant URL(s)
[6/20] Searching: site:irena.org San Marino mini-grid
  → 4 result(s), 0 new relevant URL(s)
[7/20] Searching: site:esmap.org San Marino mini-grid
  → 10 result(s), 2 new relevant URL(s)
[8/20] Searching: site:africa-energy-portal.org San Marino mini-grid
  → 10 result(s), 0 new relevant URL(s)
[9/20] Searching: San Marino mini-grid sviluppatore privato BBOXX Engie PowerGen Ignite Husk
  → 10 result(s), 0 new relevant URL(s)
[10/20] Searching: San Marino m

In [25]:
# ── Preview ───────────────────────────────────────────────────────────────

for u in urls[:20]:
    print(u)

https://www.sanmarinortv.sm/news/attualita-c4/da-san-marino-l-innovativo-generatore-portatile-green-a283339
https://www.ansa.it/canale_legalita_scuola/notizie/universita_degli_studi_della_repubblica_di_san_marino/2025/12/02/progetto-sul-biogas-premiato-alluniversita-di-san-marino_01d86efc-ecc2-49e2-8194-2ba51745812c.html
https://www.bitmat.it/case-history/repubblica-di-san-marino-modelli-innovativi-di-mobilita-sostenibile-ed-autosufficienza-energetica/
https://ansabrasil.com.br/canale_legalita_scuola/notizie/universita_degli_studi_della_repubblica_di_san_marino/2024/12/04/energia-e-risparmio-dacqua-progetto-premiato-a-san-marino_6c4a7984-9aa0-4cd9-b251-06a7529cec26.html
https://www.6wresearch.com/industry-report/san-marino-smart-microgrid-market
https://www.esmap.org/Job-Creation-in-Kosovo
https://www.esmap.org/node/3388
